# 03 · Synthetic Applicants

Builds the applicant data format as Pydantic schemas, one form at a time, then assembles a complete applicant and saves it as JSON.

In [1]:
import sys
from datetime import date
from pathlib import Path
from pydantic import ValidationError

sys.path.append("../src")  # let Python find the mortgage_agent package
from mortgage_agent.schemas import (Applicant, Assets, Credit, Debt, ExpectedOutcome,
                                    Gift, Income, LargeDeposit, Loan)

## 1. Loan

### Loan form
*The live version is in `src/mortgage_agent/schemas.py`.*

```python
class Loan(BaseModel):
    """The mortgage being requested and the property behind it"""
    amount: float = Field(gt=0, description="Loan amount in dollars")
    property_value: float = Field(gt=0, description="Appraised value in dollars")
    property_type: Literal["1-unit", "2-4 unit"]
    occupancy: Literal["principal residence", "second home", "investment property"]
    underwriting: Literal["manual", "DU"]
```

In [9]:
good = Loan(amount=320_000, property_value=400_000, monthly_housing_payment=2_420,
            property_type="1-unit", occupancy="principal residence", 
            underwriting="manual")

print(good)

amount=320000.0 property_value=400000.0 monthly_housing_payment=2420.0 property_type='1-unit' occupancy='principal residence' underwriting='manual'


## 2. Income

### Income form
*Reference copy. The live version is in `src/mortgage_agent/schemas.py`.*

```python
class Income(BaseModel):
    """What the borrower states they earn on the application"""
    employer: str = Field(min_length=1, description="Current employer name")
    monthly_income: float = Field(gt=0, description="Stated gross monthly income in dollars")
    years_employed: float = Field(ge=0, description="Years with current employer")
    pay_frequency: Literal["weekly", "biweekly", "semimonthly", "monthly"]
    self_employed: bool = False
```

In [3]:
good = Income(employer="Northwind Logistics", monthly_income=8_500,
              years_employed=3, pay_frequency="biweekly")
print(good)
print()

try:
    Income(employer="", monthly_income=8_500,
           years_employed=-1, pay_frequency="every Tuesday")
except ValidationError as e:
    print(e)

employer='Northwind Logistics' monthly_income=8500.0 years_employed=3.0 pay_frequency='biweekly' self_employed=False

3 validation errors for Income
employer
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
years_employed
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
pay_frequency
  Input should be 'weekly', 'biweekly', 'semimonthly' or 'monthly' [type=literal_error, input_value='every Tuesday', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 3. Debts

### Debt form
*Reference copy. The live version is in `src/mortgage_agent/schemas.py`.*

```python
class Debt(BaseModel):
    """One recurring monthly debt from the credit report"""
    kind: Literal["auto loan", "student loan", "credit card", "personal loan", "other"]
    monthly_payment: float = Field(ge=0, description="Required monthly payment in dollars")
    balance: float = Field(ge=0, description="Remaining balance in dollars")
    months_remaining: int | None = Field(default=None, ge=0, 
                                         description="Payments left; None for revolving debt like credit cards")
```

In [4]:
debts = [
    Debt(kind="auto loan", monthly_payment=450, balance=9_000, months_remaining=20),
    Debt(kind="student loan", monthly_payment=0, balance=30_000, months_remaining=None),
    Debt(kind="credit card", monthly_payment=120, balance=3_500),
]
for d in debts:
    print(d)

print("Total listed payments:", sum(d.monthly_payment for d in debts))

kind='auto loan' monthly_payment=450.0 balance=9000.0 months_remaining=20
kind='student loan' monthly_payment=0.0 balance=30000.0 months_remaining=None
kind='credit card' monthly_payment=120.0 balance=3500.0 months_remaining=None
Total listed payments: 570.0


## 4. Credit

### Credit form
*Reference copy. The live version is in `src/mortgage_agent/schemas.py`.*

```python
class Credit(BaseModel):
    """The borrower's credit score and major past credit events"""
    score: int = Field(ge=300, le=850, description="Representative credit score")
    bankruptcy_discharge_date: date | None = Field(
        default=None, description="Date a bankruptcy was discharged, if any"
    )
    foreclosure_date: date | None = Field(
        default=None, description="Date a foreclosure was completed, if any")
```

In [5]:
clean = Credit(score=705)
print(clean)

past_trouble = Credit(score=640, bankruptcy_discharge_date="2023-03-15")
print(past_trouble)
print(type(past_trouble.bankruptcy_discharge_date))

years_ago = (date.today() - past_trouble.bankruptcy_discharge_date).days / 365
print(f"Bankruptcy discharged {years_ago:.1f} years ago")

try:
    Credit(score=900, foreclosure_date="last summer")
except ValidationError as e:
    print(e)

score=705 bankruptcy_discharge_date=None foreclosure_date=None
score=640 bankruptcy_discharge_date=datetime.date(2023, 3, 15) foreclosure_date=None
<class 'datetime.date'>
Bankruptcy discharged 3.6 years ago
2 validation errors for Credit
score
  Input should be less than or equal to 850 [type=less_than_equal, input_value=900, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
foreclosure_date
  Input should be a valid date or datetime, invalid character in year [type=date_from_datetime_parsing, input_value='last summer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/date_from_datetime_parsing


## 5. Assets

### LargeDeposit, Gift, and Assets forms
*Reference copy. The live version is in `src/mortgage_agent/schemas.py`.*

```python
class LargeDeposit(BaseModel):
    """A single large deposit found on a bank statement"""
    amount: float = Field(gt=0, description="Deposit amount in dollars")
    deposit_date: date
    source_documented: bool = Field(description="Whether the source of the funds is explained and documented")

class Gift(BaseModel):
    """Money given to the borrower to help with the purchase"""
    amount: float = Field(gt=0, description="Gift amount in dollars")
    donor_relationship: Literal["parent", "grandparent", "sibling", "spouse",
                                "fiance", "other relative", "friend", "employer"]
    has_gift_letter: bool

class Assets(BaseModel):
    """The borrower's funds available for closing and reserves"""
    bank_balance: float = Field(ge=0, description="Checking and savings balance in dollars")
    retirement_balance: float = Field(default=0.0, ge=0, description="401(k)/IRA balance in dollars")
    large_deposits: list[LargeDeposit] = Field(default_factory=list)
    gifts: list[Gift] = Field(default_factory=list)
```

In [6]:
sarah_assets = Assets(
    bank_balance=95_000,
    large_deposits=[LargeDeposit(amount=40_000, deposit_date="2026-09-01", source_documented=False)],
    gifts=[Gift(amount=30_000, donor_relationship="parent", has_gift_letter=True)],
)

print(sarah_assets)

simple_assets = Assets(bank_balance=50_000)
print(simple_assets)

bank_balance=95000.0 retirement_balance=0.0 large_deposits=[LargeDeposit(amount=40000.0, deposit_date=datetime.date(2026, 9, 1), source_documented=False)] gifts=[Gift(amount=30000.0, donor_relationship='parent', has_gift_letter=True)]
bank_balance=50000.0 retirement_balance=0.0 large_deposits=[] gifts=[]


## 6. Answer Key and Complete Applicant

### ExpectedOutcome and Applicant forms
*Reference copy. The live version is in `src/mortgage_agent/schemas.py`.*

```python
class ExpectedOutcome(BaseModel):
    """The correct underwriting decision, used only for grading. Never shown to the agent."""
    decision: Literal["approve", "approve with conditions", "decline"]
    reasons: list[str]
    rule_ids: list[str] = Field(description="Selling Guide rules that drive the decision")

class Applicant(BaseModel):
    """A complete synthetic mortgage application"""
    applicant_id: str
    name: str
    loan: Loan
    income: Income
    debts: list[Debt] = Field(default_factory=list)
    credit: Credit
    assets: Assets
    expected: ExpectedOutcome
```

In [7]:
sarah = Applicant(
    applicant_id="A007",
    name="Sarah Chen",
    loan=Loan(amount=320_000, property_value=400_000, monthly_housing_payment=2_420,
              property_type="1-unit", occupancy="principal residence", underwriting="manual"),
    income=Income(employer="Northwind Logistics", monthly_income=8_500,
                  years_employed=3, pay_frequency="biweekly"),
    debts=[Debt(kind="auto loan", monthly_payment=450, balance=9_000, months_remaining=20),
           Debt(kind="credit card", monthly_payment=120, balance=3_500)],
    credit=Credit(score=705),
    assets=Assets(bank_balance=95_000,
                  large_deposits=[LargeDeposit(amount=40_000, deposit_date="2026-09-01",
                                               source_documented=False)]),
    expected=ExpectedOutcome(
        decision="approve with conditions",
        reasons=["Undocumented $40,000 deposit must be sourced"],
        rule_ids=["B3-4.2-02"],
    ),
)

# Save to disk
out_dir = Path("../data/applicants")
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"{sarah.applicant_id}.json"
path.write_text(sarah.model_dump_json(indent=2))

# Load it back 
loaded = Applicant.model_validate_json(path.read_text())
print("Round trip identical:", loaded == sarah)

# What the agent will see: everything EXCEPT the answer key
agent_view = loaded.model_dump(exclude={"expected"})
print("Agent sees:", list(agent_view.keys()))

Round trip identical: True
Agent sees: ['applicant_id', 'name', 'loan', 'income', 'debts', 'credit', 'assets']


## 7. Load an Applicant

In [8]:
sarah = Applicant.model_validate_json(Path("../data/applicants/A007.json").read_text())
print(sarah.name, "|", sarah.loan.amount, "|", sarah.expected.decision)

Sarah Chen | 320000.0 | approve with conditions
